In [4]:
#!/usr/bin/env python3
"""
diagnostic_photometry_offset.py
Identifica la causa del offset sistemático entre SPLUS y DECam
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

def analyze_photometry_offset(splus_catalog, taylor_catalog, output_dir="offset_analysis"):
    """Análisis completo del offset fotométrico"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Mapeo de filtros
    filter_mapping = {
        'F378': 'umag', 'F395': 'umag', 'F410': 'gmag', 
        'F430': 'gmag', 'F515': 'gmag', 'F660': 'rmag', 'F861': 'imag'
    }
    
    # Análisis por filtro
    offset_results = {}
    
    for splus_filter, taylor_filter in filter_mapping.items():
        splus_col = f'MAG_{splus_filter}_2'  # Apertura de 2"
        
        if splus_col in splus_catalog.columns and taylor_filter in taylor_catalog.columns:
            # Combinar datos
            merged = pd.merge(
                splus_catalog[['T17ID', splus_col]],
                taylor_catalog[['T17ID', taylor_filter]],
                on='T17ID',
                how='inner'
            )
            
            # Filtrar valores válidos
            valid_mask = (
                (merged[splus_col] < 50) & 
                (merged[taylor_filter] < 50) &
                np.isfinite(merged[splus_col]) & 
                np.isfinite(merged[taylor_filter])
            )
            
            valid_data = merged[valid_mask]
            
            if len(valid_data) > 10:
                differences = valid_data[splus_col] - valid_data[taylor_filter]
                
                offset_results[splus_filter] = {
                    'taylor_filter': taylor_filter,
                    'n_sources': len(valid_data),
                    'mean_diff': np.mean(differences),
                    'median_diff': np.median(differences),
                    'std_diff': np.std(differences),
                    'mad_diff': stats.median_abs_deviation(differences),
                    'min_diff': np.min(differences),
                    'max_diff': np.max(differences),
                    'data': valid_data
                }
    
    # Crear visualizaciones
    plot_offset_analysis(offset_results, output_dir)
    
    return offset_results

def plot_offset_analysis(offset_results, output_dir):
    """Visualización del análisis de offsets"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Histograma de offsets por filtro
    colors = plt.cm.Set3(np.linspace(0, 1, len(offset_results)))
    
    for i, (filt, results) in enumerate(offset_results.items()):
        differences = results['data'][f'MAG_{filt}_2'] - results['data'][results['taylor_filter']]
        axes[0, 0].hist(differences, bins=30, alpha=0.7, color=colors[i], 
                       label=f'{filt} vs {results["taylor_filter"]}', density=True)
    
    axes[0, 0].axvline(0, color='black', linestyle='--', alpha=0.8)
    axes[0, 0].set_xlabel('Δmag (SPLUS - DECam)')
    axes[0, 0].set_ylabel('Density')
    axes[0, 0].set_title('Distribución de Offsets por Filtro')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Resumen estadístico
    filters = list(offset_results.keys())
    medians = [results['median_diff'] for results in offset_results.values()]
    stds = [results['std_diff'] for results in offset_results.values()]
    
    axes[0, 1].bar(filters, medians, yerr=stds, capsize=5, alpha=0.7, color='skyblue')
    axes[0, 1].axhline(0, color='black', linestyle='--', alpha=0.8)
    axes[0, 1].set_xlabel('Filtro SPLUS')
    axes[0, 1].set_ylabel('Δmag Mediana (SPLUS - DECam)')
    axes[0, 1].set_title('Offset Sistemático por Filtro')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Correlación magnitud vs offset
    for i, (filt, results) in enumerate(offset_results.items()):
        taylor_mags = results['data'][results['taylor_filter']]
        differences = results['data'][f'MAG_{filt}_2'] - taylor_mags
        
        axes[1, 0].scatter(taylor_mags, differences, alpha=0.6, 
                          color=colors[i], label=filt, s=20)
    
    axes[1, 0].axhline(0, color='black', linestyle='--', alpha=0.8)
    axes[1, 0].set_xlabel('Magnitud DECam')
    axes[1, 0].set_ylabel('Δmag (SPLUS - DECam)')
    axes[1, 0].set_title('Dependencia del Offset con la Magnitud')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].invert_xaxis()
    
    # 4. Mapa de offsets por filtro (boxplot)
    data_to_plot = []
    labels = []
    for filt, results in offset_results.items():
        differences = results['data'][f'MAG_{filt}_2'] - results['data'][results['taylor_filter']]
        data_to_plot.append(differences)
        labels.append(f'{filt}\nvs\n{results["taylor_filter"]}')
    
    axes[1, 1].boxplot(data_to_plot, labels=labels)
    axes[1, 1].axhline(0, color='black', linestyle='--', alpha=0.8)
    axes[1, 1].set_ylabel('Δmag (SPLUS - DECam)')
    axes[1, 1].set_title('Distribución de Offsets (Boxplot)')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/offset_analysis_comprehensive.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    # Crear tabla resumen
    summary_data = []
    for filt, results in offset_results.items():
        summary_data.append({
            'SPLUS_Filter': filt,
            'DECam_Filter': results['taylor_filter'],
            'N_Sources': results['n_sources'],
            'Median_Offset': results['median_diff'],
            'Mean_Offset': results['mean_diff'],
            'Std_Offset': results['std_diff'],
            'MAD_Offset': results['mad_diff']
        })
    
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(f'{output_dir}/offset_analysis_summary.csv', index=False)
    
    print("📊 ANÁLISIS DE OFFSETS COMPLETADO")
    print(summary_df.to_string(index=False))

def test_aperture_correction_direction(splus_catalog, taylor_catalog):
    """Prueba específica para la dirección de la corrección de apertura"""
    
    print("🧪 TESTING APERTURE CORRECTION DIRECTION")
    
    # Seleccionar un filtro representativo
    test_filter = 'F660'
    taylor_filter = 'rmag'
    
    splus_col = f'MAG_{test_filter}_2'
    
    if splus_col in splus_catalog.columns and taylor_filter in taylor_catalog.columns:
        merged = pd.merge(
            splus_catalog[['T17ID', splus_col]],
            taylor_catalog[['T17ID', taylor_filter]],
            on='T17ID'
        )
        
        valid_mask = (
            (merged[splus_col] < 50) & 
            (merged[taylor_filter] < 50) &
            np.isfinite(merged[splus_col]) & 
            np.isfinite(merged[taylor_filter])
        )
        
        valid_data = merged[valid_mask]
        differences = valid_data[splus_col] - valid_data[taylor_filter]
        
        median_offset = np.median(differences)
        
        print(f"📐 Offset mediano ({test_filter} vs {taylor_filter}): {median_offset:.3f} mag")
        
        if median_offset > 0:
            print("❌ SPLUS es MÁS DÉBIL que DECam")
            print("   Esto sugiere que la corrección de apertura podría estar aplicándose al revés")
        else:
            print("✅ SPLUS es MÁS BRILLANTE que DECam")
            print("   Esto coincide con tu observación")
        
        return median_offset

def main():
    """Función principal de diagnóstico"""
    
    # Cargar catálogos
    splus_catalog = pd.read_csv("../anac_data/Results/all_fields_gc_photometry_corrected_errors_v17.csv")
    taylor_catalog = pd.read_csv("../TAP_1_J_MNRAS_3444_gc.csv")
    
    print("🔍 INICIANDO DIAGNÓSTICO DE OFFSETS FOTOMÉTRICOS")
    print("=" * 60)
    
    # 1. Análisis completo de offsets
    offset_results = analyze_photometry_offset(splus_catalog, taylor_catalog)
    
    # 2. Prueba específica de dirección
    test_aperture_correction_direction(splus_catalog, taylor_catalog)
    
    # 3. Análisis por rango de magnitud
    print("\n📈 ANALIZANDO OFFSETS POR RANGO DE MAGNITUD:")
    
    for splus_filter in ['F660', 'F861']:  # Filtros representativos
        splus_col = f'MAG_{splus_filter}_2'
        taylor_filter = 'rmag' if splus_filter == 'F660' else 'imag'
        
        if splus_col in splus_catalog.columns and taylor_filter in taylor_catalog.columns:
            merged = pd.merge(
                splus_catalog[['T17ID', splus_col]],
                taylor_catalog[['T17ID', taylor_filter]],
                on='T17ID'
            )
            
            valid_mask = (
                (merged[splus_col] < 50) & 
                (merged[taylor_filter] < 50) &
                np.isfinite(merged[splus_col]) & 
                np.isfinite(merged[taylor_filter])
            )
            
            valid_data = merged[valid_mask]
            
            # Dividir en rangos de magnitud
            mag_bins = [18, 20, 22, 24, 26]
            for i in range(len(mag_bins)-1):
                bin_mask = (valid_data[taylor_filter] >= mag_bins[i]) & (valid_data[taylor_filter] < mag_bins[i+1])
                bin_data = valid_data[bin_mask]
                
                if len(bin_data) > 5:
                    differences = bin_data[splus_col] - bin_data[taylor_filter]
                    print(f"  {splus_filter} vs {taylor_filter} [{mag_bins[i]}-{mag_bins[i+1]}]: "
                          f"Δmediana = {np.median(differences):.3f}, n = {len(bin_data)}")

if __name__ == "__main__":
    main()

🔍 INICIANDO DIAGNÓSTICO DE OFFSETS FOTOMÉTRICOS


/tmp/ipykernel_1646280/3326660640.py:127: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[1, 1].boxplot(data_to_plot, labels=labels)


📊 ANÁLISIS DE OFFSETS COMPLETADO
SPLUS_Filter DECam_Filter  N_Sources  Median_Offset  Mean_Offset  Std_Offset  MAD_Offset
        F378         umag       2509      -1.034650     0.079815    5.246180    0.707620
        F395         umag       2455      -1.116546     0.113776    5.539709    0.743396
        F410         gmag       2711       0.224753     1.457649    5.073550    0.601854
        F430         gmag       2789       0.277582     1.424704    5.020267    0.550524
        F515         gmag       3110      -0.254968     0.915569    5.063623    0.505604
        F660         rmag       3312      -0.045983     1.115410    4.787690    0.152453
        F861         imag       3293       0.005309     1.093999    4.681345    0.210935
🧪 TESTING APERTURE CORRECTION DIRECTION
📐 Offset mediano (F660 vs rmag): -0.046 mag
✅ SPLUS es MÁS BRILLANTE que DECam
   Esto coincide con tu observación

📈 ANALIZANDO OFFSETS POR RANGO DE MAGNITUD:
  F660 vs rmag [18-20]: Δmediana = -0.042, n = 1013
  F

# Corrections

In [7]:
#!/usr/bin/env python3
"""
correct_photometry_offset.py
Corrige los offsets sistemáticos identificados en el diagnóstico
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Offsets específicos por filtro basados en tu diagnóstico
FILTER_OFFSETS = {
    'F378': +1.035,  # SPLUS más brillante, necesitamos hacerlo más débil
    'F395': +1.117,  # SPLUS más brillante  
    'F410': -0.225,  # SPLUS más débil, necesitamos hacerlo más brillante
    'F430': -0.278,  # SPLUS más débil
    'F515': +0.255,  # SPLUS más brillante
    'F660': +0.046,  # SPLUS más brillante
    'F861': -0.005   # Casi neutro
}

def apply_cross_calibration(splus_catalog, output_path):
    """Aplica calibración cruzada basada en el análisis de offsets"""
    
    print("🔧 APLICANDO CALIBRACIÓN CRUZADA SPLUS-DECam")
    
    # Crear copia del catálogo
    calibrated_catalog = splus_catalog.copy()
    
    # Aplicar offsets a cada filtro y apertura
    apertures = ['2', '3']  # Tus aperturas principales
    
    for filter_name, offset in FILTER_OFFSETS.items():
        for aperture in apertures:
            mag_col = f'MAG_{filter_name}_{aperture}'
            flux_col = f'FLUX_{filter_name}_{aperture}'
            
            if mag_col in calibrated_catalog.columns:
                print(f"  Calibrando {mag_col}: offset = {offset:.3f} mag")
                
                # Aplicar offset a magnitudes
                valid_mask = calibrated_catalog[mag_col] < 50
                calibrated_catalog.loc[valid_mask, mag_col] = (
                    calibrated_catalog.loc[valid_mask, mag_col] - offset
                )
                
                # Actualizar flujos si existen
                if flux_col in calibrated_catalog.columns:
                    calibrated_catalog.loc[valid_mask, flux_col] = (
                        calibrated_catalog.loc[valid_mask, flux_col] * 10**(0.4 * offset)
                    )
    
    # Guardar catálogo calibrado
    calibrated_catalog.to_csv(output_path, index=False)
    print(f"✅ Catálogo calibrado guardado: {output_path}")
    
    return calibrated_catalog

def validate_calibration(original_catalog, calibrated_catalog, taylor_catalog):
    """Valida la efectividad de la calibración"""
    
    print("\n📊 VALIDANDO CALIBRACIÓN")
    
    filter_mapping = {
        'F378': 'umag', 'F395': 'umag', 'F410': 'gmag', 
        'F430': 'gmag', 'F515': 'gmag', 'F660': 'rmag', 'F861': 'imag'
    }
    
    validation_results = {}
    
    for splus_filter, taylor_filter in filter_mapping.items():
        orig_col = f'MAG_{splus_filter}_2'
        calib_col = f'MAG_{splus_filter}_2'
        
        if orig_col in original_catalog.columns and calib_col in calibrated_catalog.columns:
            # Original vs Taylor
            merged_orig = pd.merge(
                original_catalog[['T17ID', orig_col]],
                taylor_catalog[['T17ID', taylor_filter]],
                on='T17ID'
            )
            valid_orig = merged_orig[(merged_orig[orig_col] < 50) & (merged_orig[taylor_filter] < 50)]
            diff_orig = valid_orig[orig_col] - valid_orig[taylor_filter]
            
            # Calibrado vs Taylor
            merged_calib = pd.merge(
                calibrated_catalog[['T17ID', calib_col]],
                taylor_catalog[['T17ID', taylor_filter]],
                on='T17ID'
            )
            valid_calib = merged_calib[(merged_calib[calib_col] < 50) & (merged_calib[taylor_filter] < 50)]
            diff_calib = valid_calib[calib_col] - valid_calib[taylor_filter]
            
            if len(diff_orig) > 10 and len(diff_calib) > 10:
                validation_results[splus_filter] = {
                    'original_median': np.median(diff_orig),
                    'calibrated_median': np.median(diff_calib),
                    'improvement': np.median(diff_orig) - np.median(diff_calib),
                    'n_sources': len(diff_calib)
                }
    
    # Mostrar resultados
    print("Resultados de la validación:")
    print("Filtro  |  Original  |  Calibrado  |  Mejora")
    print("--------|------------|-------------|---------")
    for filt, results in validation_results.items():
        print(f"{filt:6} | {results['original_median']:8.3f}   | {results['calibrated_median']:8.3f}    | {results['improvement']:6.3f}")
    
    return validation_results

def create_calibration_plot(validation_results, output_path):
    """Crea gráfico comparativo antes/después de la calibración"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    filters = list(validation_results.keys())
    original_offsets = [results['original_median'] for results in validation_results.values()]
    calibrated_offsets = [results['calibrated_median'] for results in validation_results.values()]
    
    # Gráfico de barras comparativo
    x_pos = np.arange(len(filters))
    width = 0.35
    
    ax1.bar(x_pos - width/2, original_offsets, width, label='Original', alpha=0.7, color='red')
    ax1.bar(x_pos + width/2, calibrated_offsets, width, label='Calibrado', alpha=0.7, color='green')
    
    ax1.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax1.set_xlabel('Filtro')
    ax1.set_ylabel('Δmag Mediana (SPLUS - DECam)')
    ax1.set_title('Comparación Antes/Después de Calibración')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(filters)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Gráfico de mejoras
    improvements = [results['improvement'] for results in validation_results.values()]
    ax2.bar(filters, improvements, alpha=0.7, color='blue')
    ax2.set_xlabel('Filtro')
    ax2.set_ylabel('Mejora (mag)')
    ax2.set_title('Mejora en la Calibración por Filtro')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"📈 Gráfico de validación guardado: {output_path}")

def main():
    """Función principal de calibración"""
    
    # Archivos de entrada
    splus_catalog_path = "../anac_data/Results/all_fields_gc_photometry_corrected_errors_v17.csv"
    taylor_catalog_path = "../TAP_1_J_MNRAS_3444_gc.csv"
    output_catalog_path = "../anac_data/Results/gc_photometry_cross_calibrated_v18.csv"
    
    # Cargar catálogos
    print("📁 Cargando catálogos...")
    splus_catalog = pd.read_csv(splus_catalog_path)
    taylor_catalog = pd.read_csv(taylor_catalog_path)
    
    # Aplicar calibración cruzada
    calibrated_catalog = apply_cross_calibration(splus_catalog, output_catalog_path)
    
    # Validar calibración
    validation_results = validate_calibration(splus_catalog, calibrated_catalog, taylor_catalog)
    
    # Crear gráfico de validación
    create_calibration_plot(validation_results, "calibration_validation.png")
    
    print("\n🎉 CALIBRACIÓN COMPLETADA")
    print("Los offsets sistemáticos han sido corregidos usando:")
    print("   - Análisis empírico de diferencias SPLUS vs DECam")
    print("   - Offsets específicos por filtro")
    print("   - Validación con catálogo de referencia Taylor et al.")

if __name__ == "__main__":
    main()

📁 Cargando catálogos...
🔧 APLICANDO CALIBRACIÓN CRUZADA SPLUS-DECam
  Calibrando MAG_F378_2: offset = 1.035 mag
  Calibrando MAG_F378_3: offset = 1.035 mag
  Calibrando MAG_F395_2: offset = 1.117 mag
  Calibrando MAG_F395_3: offset = 1.117 mag
  Calibrando MAG_F410_2: offset = -0.225 mag
  Calibrando MAG_F410_3: offset = -0.225 mag
  Calibrando MAG_F430_2: offset = -0.278 mag
  Calibrando MAG_F430_3: offset = -0.278 mag
  Calibrando MAG_F515_2: offset = 0.255 mag
  Calibrando MAG_F515_3: offset = 0.255 mag
  Calibrando MAG_F660_2: offset = 0.046 mag
  Calibrando MAG_F660_3: offset = 0.046 mag
  Calibrando MAG_F861_2: offset = -0.005 mag
  Calibrando MAG_F861_3: offset = -0.005 mag
✅ Catálogo calibrado guardado: ../anac_data/Results/gc_photometry_cross_calibrated_v18.csv

📊 VALIDANDO CALIBRACIÓN
Resultados de la validación:
Filtro  |  Original  |  Calibrado  |  Mejora
--------|------------|-------------|---------
F378   |   -1.035   |   -2.070    |  1.035
F395   |   -1.117   |   -2.234 